<a id="top"></a>

# Demo: schemas as a teaching tool

```yaml
title: "Demo: schemas as a teaching tool"
keywords: parameter schema, loose vs tight schema, validation error, informative vs opaque, shape not truth, book_meeting, recoverable error, teacher demo
estimated duration: 14
```

> **Lesson:** L08. Teacher demo — Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L08/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Clear outputs before committing.
>
> **Why the raw Anthropic SDK here, not `potato_llm`?** The course's `potato_llm` seam is **text-only** — its `Message` cannot represent `tool_use` / `tool_result` blocks. L08 registers tools and watches the model *choose* and *call* them, so these demos call the raw SDK directly (exactly as [L07](../L07/L0701_intro.md) introduced). The API key still loads through the config seam (`require_anthropic_key`); we never hard-code it. L08 changes the *client*, not where secrets come from.
>
> The point to land: **a tight schema converts ambiguity into validation errors the model can recover from; a loose schema pushes the ambiguity into a silent runtime guess.** And the error *shape* decides whether the model recovers — an error is a prompt for its next turn.

## Contents

- [1. Setup — loose vs tight book_meeting, and two error styles](#1-setup--loose-vs-tight-book_meeting-and-two-error-styles)
- [2. The loose schema — silent wrongness](#2-the-loose-schema--silent-wrongness)
- [3. Tight schema + opaque errors — a noisy, unhelpful loop](#3-tight-schema--opaque-errors--a-noisy-unhelpful-loop)
- [4. Tight schema + informative errors — failing productively](#4-tight-schema--informative-errors--failing-productively)
- [5. Schemas validate shape, not truth](#5-schemas-validate-shape-not-truth)
- [6. Takeaways](#6-takeaways)

## 1. Setup — loose vs tight book_meeting, and two error styles

One conceptual tool, `book_meeting`, in two schema shapes: **loose** (one free-form `details` string) and **tight** (typed, all-required fields with per-parameter constraints). The tight tool's validator comes in two styles — **informative** errors that name the field + constraint, and **opaque** errors that don't. One ambiguous prompt drives all three runs. Anchor model: **Claude Sonnet 4.6**.

In [ ]:
import anthropic

from fluffy_potato_curriculum.common.config import require_anthropic_key

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic(api_key=require_anthropic_key())


LOOSE_TOOL: dict[str, object] = {
    "name": "book_meeting",
    "description": "Book a meeting.",
    "input_schema": {
        "type": "object",
        "properties": {"details": {"type": "string", "description": "the meeting details"}},
        "required": ["details"],
    },
}
TIGHT_TOOL: dict[str, object] = {
    "name": "book_meeting",
    "description": (
        "Book a calendar meeting. All fields are required and validated; on a bad field you "
        "will receive a structured validation error naming the field to fix."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "attendee_email": {
                "type": "string",
                "description": "RFC 5322 email, e.g. 'priya@example.com'.",
            },
            "start_iso": {
                "type": "string",
                "description": "ISO 8601 start, e.g. '2024-11-05T14:00:00'.",
            },
            "duration_minutes": {"type": "integer", "description": "integer between 15 and 240."},
            "title": {"type": "string", "description": "short meeting title."},
        },
        "required": ["attendee_email", "start_iso", "duration_minutes", "title"],
    },
}


def validate_informative(args: dict[str, object]) -> dict[str, object]:
    """Validate tight-schema args; return a structured {error, field, message} or a booking."""
    email = str(args.get("attendee_email", ""))
    if "@" not in email:
        return {
            "error": "validation",
            "field": "attendee_email",
            "message": f"must be an email, got {email!r}",
        }
    minutes = args.get("duration_minutes")
    if not isinstance(minutes, int) or not (15 <= minutes <= 240):
        return {
            "error": "validation",
            "field": "duration_minutes",
            "message": f"must be an integer 15..240, got {minutes!r}",
        }
    return {"booked": True, "with": email, "minutes": minutes}


def validate_opaque(args: dict[str, object]) -> dict[str, object]:
    """Same checks, but every failure collapses to a useless generic message."""
    res = validate_informative(args)
    if "error" in res:
        return {"error": "bad input"}  # no field, no constraint
    return res


def show_turn(resp) -> None:
    print("stop_reason:", resp.stop_reason)
    for b in resp.content:
        if b.type == "tool_use":
            print(f"  tool_use  {b.name}  args={b.input}")
        else:
            print(f"  text      {getattr(b, 'text', '')!r}")


# Ambiguous on purpose: no email, vague "afternoon", relative date.
PROMPT = "Book a 90-minute design review with Priya next Tuesday afternoon."
print("setup ready (live model:", MODEL, ")")

[↑ Back to top](#top)

## 2. The loose schema — silent wrongness

Run the ambiguous prompt against the **loose** schema. The model packs everything into the `details` string and the tool 'succeeds'. Whether the meeting was booked *correctly* — with whose email, at what exact time — is **impossible to tell** from the conversation. This is the worst outcome: it looks like it worked.

In [ ]:
resp = client.messages.create(
    model=MODEL,
    max_tokens=400,
    tools=[LOOSE_TOOL],
    messages=[{"role": "user", "content": PROMPT}],
)
show_turn(resp)
use = next((b for b in resp.content if b.type == "tool_use"), None)
if use is not None:
    print("\nthe tool got ONE blob:", use.input)
    print("booked correctly? impossible to tell from here.")

[↑ Back to top](#top)

## 3. Tight schema + opaque errors — a noisy, unhelpful loop

Run the same prompt against the **tight** schema with **opaque** errors. The model guesses each field and submits; the validator returns `{'error': 'bad input'}`. We hand that back and let the model try again — it typically retries with *another* guess, still opaque, still wrong. We run two rounds to show the loop.

In [ ]:
messages: list[dict[str, object]] = [{"role": "user", "content": PROMPT}]
for round_num in range(2):
    resp = client.messages.create(
        model=MODEL, max_tokens=400, tools=[TIGHT_TOOL], messages=messages
    )
    print(f"--- round {round_num + 1} ---")
    show_turn(resp)
    use = next((b for b in resp.content if b.type == "tool_use"), None)
    if use is None:
        break
    result = validate_opaque(use.input)
    print("  tool returned:", result)
    messages.append({"role": "assistant", "content": resp.content})
    messages.append(
        {
            "role": "user",
            "content": [{"type": "tool_result", "tool_use_id": use.id, "content": str(result)}],
        }
    )

[↑ Back to top](#top)

## 4. Tight schema + informative errors — failing productively

Same prompt, same tight schema, but now **informative** errors. The model submits, gets a structured error naming the offending field, and on the next turn either **asks the user** for the missing detail (Priya's email) or **fixes the field** and retries. The error taught the model what to do.

In [ ]:
messages = [{"role": "user", "content": PROMPT}]
for round_num in range(3):
    resp = client.messages.create(
        model=MODEL, max_tokens=400, tools=[TIGHT_TOOL], messages=messages
    )
    print(f"--- round {round_num + 1} ---")
    show_turn(resp)
    use = next((b for b in resp.content if b.type == "tool_use"), None)
    if use is None:
        print("  (model stopped calling — asked the user or finished)")
        break
    result = validate_informative(use.input)
    print("  tool returned:", result)
    messages.append({"role": "assistant", "content": resp.content})
    messages.append(
        {
            "role": "user",
            "content": [{"type": "tool_result", "tool_use_id": use.id, "content": str(result)}],
        }
    )
    if "booked" in result:
        break

[↑ Back to top](#top)

## 5. Schemas validate shape, not truth

A subtle trap to name out loud: the model may guess `priya@example.com` so confidently that it **passes** email-format validation — even though no such user may exist. The tight schema catches *malformed* arguments; it does **not** catch *false* ones. That gap is exactly why **runtime errors** ([L0809](L0809_lecture.ipynb)) are the second line of defense.

In [ ]:
fake_but_well_formed = {
    "attendee_email": "priya@example.com",  # valid shape, possibly nonexistent person
    "start_iso": "2024-11-05T14:00:00",
    "duration_minutes": 90,
    "title": "Design review",
}
print("validation says:", validate_informative(fake_but_well_formed))
print("note: the shape is valid even if the address is fake — shape != truth.")

[↑ Back to top](#top)

## 6. Takeaways

- The **loose** schema *appears* to work and is the worst outcome (silent wrongness). The tight schema with **opaque** errors fails noisily but unhelpfully. The tight schema with **informative** errors fails *productively* — the model learns from each turn.
- A tight schema — required fields, narrow types, per-field constraints — turns the model's fuzziness ([L02](../L02/L0201_intro.md)) into **recoverable validation errors**.
- An **error message is a prompt** for the model's next turn — write it like a system message, not a Python traceback. The [L0808 lab](L0808_lab_empty.ipynb) drills exactly this rewrite, offline.
- Schemas validate **shape, not truth** — the bridge to [L0809](L0809_lecture.ipynb)'s runtime errors and side effects.

[↑ Back to top](#top)